In [1]:
"""
icosahedron_cube_fixed.py
=========================
Fixed cube detection: enumerate all 8-subsets and check the
distance pattern 3-3-1 (3 at edge, 3 at face diagonal, 1 at space diagonal).

Also computes edge ratios and checks against 2^(±1/3).
"""

import numpy as np
from itertools import combinations, product

phi = (1 + np.sqrt(5)) / 2
target_cos = 2 ** (-1/3)

print("=" * 70)
print("ICOSAHEDRON: FIXED CUBE DETECTION")
print("=" * 70)
print()

# Build icosahedron
verts = []
for s1, s2 in product([1, -1], repeat=2):
    verts.append([0, s1, s2 * phi])
    verts.append([s1, s2 * phi, 0])
    verts.append([s2 * phi, 0, s1])

unique = []
for v in verts:
    if not any(np.allclose(v, u) for u in unique):
        unique.append(v)
verts = np.array(unique)

R = np.linalg.norm(verts[0])
edge_ico = 2.0  # known

print(f"Icosahedron:")
print(f"  R (radius)      = {R:.10f}")
print(f"  edge            = {edge_ico:.10f}")
print(f"  edge / R        = {edge_ico/R:.10f}")
print()

# ============================================================
# CUBE DETECTION
# ============================================================
def check_cube(indices, verts, tol=1e-5):
    """
    Check if 8 vertices form a cube.
    Returns cube edge length, or None.
    """
    sub = verts[list(indices)]
    n = 8

    # Pairwise distances
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            d = np.linalg.norm(sub[i] - sub[j])
            D[i, j] = d
            D[j, i] = d

    # Minimum positive distance = cube edge candidate
    pos = D[D > 0]
    c = pos.min()

    # Check distance pattern 3-3-1 from each vertex
    for i in range(n):
        n_c = 0    # at distance c (edge)
        n_c2 = 0   # at distance c√2 (face diagonal)
        n_c3 = 0   # at distance c√3 (space diagonal)
        for j in range(n):
            if i == j:
                continue
            d = D[i, j]
            if abs(d - c) < tol * c:
                n_c += 1
            elif abs(d - c * np.sqrt(2)) < tol * c:
                n_c2 += 1
            elif abs(d - c * np.sqrt(3)) < tol * c:
                n_c3 += 1
            else:
                return None
        if (n_c, n_c2, n_c3) != (3, 3, 1):
            return None

    return c


cubes = []
for indices in combinations(range(12), 8):
    c = check_cube(indices, verts)
    if c is not None:
        cubes.append((indices, c))

print(f"Cubes found: {len(cubes)}")
print()

if cubes:
    for idx, (indices, c) in enumerate(cubes):
        ratio_to_ico = c / edge_ico
        print(f"Cube {idx+1}:")
        print(f"  vertices: {indices}")
        print(f"  cube edge        = {c:.10f}")
        print(f"  cube / ico edge  = {ratio_to_ico:.10f}")
        print(f"  (cube/ico)^3     = {ratio_to_ico**3:.10f}")
        print()

    # Report ratios
    ratios = [c / edge_ico for _, c in cubes]
    print("=" * 70)
    print("EDGE RATIOS")
    print("=" * 70)
    print(f"  All ratios equal?   {len(set(round(r, 10) for r in ratios)) == 1}")
    print(f"  Ratio value:        {ratios[0]:.10f}")
    print()
    print("Compare with fundamental constants:")
    print(f"  φ              = {phi:.10f}")
    print(f"  1/φ            = {1/phi:.10f}")
    print(f"  φ²             = {phi**2:.10f}")
    print(f"  √(2+φ)         = {np.sqrt(2+phi):.10f}")
    print(f"  √(3-φ)         = {np.sqrt(3-phi):.10f}")
    print(f"  2^(1/3)        = {2**(1/3):.10f}")
    print(f"  2^(-1/3)       = {2**(-1/3):.10f}")
    print(f"  2^(1/6)        = {2**(1/6):.10f}")
    print(f"  √2             = {np.sqrt(2):.10f}")
    print(f"  √3             = {np.sqrt(3):.10f}")
    print()

    # Match check
    r = ratios[0]
    print("Possible matches for the ratio:")
    candidates = {
        "2^(1/3)":   2**(1/3),
        "2^(2/3)":   2**(2/3),
        "2^(-1/3)":  2**(-1/3),
        "1/φ":       1/phi,
        "φ":         phi,
        "√(2/3)":    np.sqrt(2/3),
        "√(3/2)":    np.sqrt(3/2),
        "2/√3":      2/np.sqrt(3),
        "√2/φ":      np.sqrt(2)/phi,
        "φ/√2":      phi/np.sqrt(2),
    }
    for name, val in candidates.items():
        err = abs(r - val) / r
        marker = "  <<< MATCH" if err < 1e-4 else ""
        print(f"  {name:12s} = {val:.10f}  (error: {err:.2e}){marker}")
    print()

    # Check if any power gives 2
    print("Powers of the ratio:")
    for p in [1, 2, 3, 4, 6, 12]:
        print(f"  ratio^{p:<2d} = {r**p:.10f}")
    print()

else:
    print("No cubes found — this should not happen for an icosahedron.")
    print("The icosahedron is known to contain exactly 5 inscribed cubes.")

ICOSAHEDRON: FIXED CUBE DETECTION

Icosahedron:
  R (radius)      = 1.9021130326
  edge            = 2.0000000000
  edge / R        = 1.0514622242

Cubes found: 0

No cubes found — this should not happen for an icosahedron.
The icosahedron is known to contain exactly 5 inscribed cubes.
